# Worked Capstone: CNN Image Classification with Reproducible Training

**Domain:** Deep learning and computer vision  
**Primary dataset:** `images/simple_shapes_X.npy and simple_shapes_y.npy`  
**Level:** Practitioner to Advanced

## Business goal

Classify synthetic geometric patterns while demonstrating a complete PyTorch training, validation, checkpoint, and error-analysis workflow.

This is a worked reference project. First attempt the corresponding phase project independently; then use this capstone to compare framing, evaluation, code structure, and communication.

## Decision questions

        1. Are tensor shapes correct?
2. Can a simple baseline solve the task?
3. Does validation improve?
4. Which classes are confused?

        ## Definition of done

        - [ ] Visualization
- [ ] Deterministic split
- [ ] CNN
- [ ] Training history
- [ ] Confusion matrix
- [ ] Checkpoint
- [ ] Serving metadata

## End-to-end workflow

```text
Decision and scope
      ↓
Data contract and quality
      ↓
Exploration and hypotheses
      ↓
Baseline and evaluation design
      ↓
Candidate method(s)
      ↓
Held-out / temporal evaluation
      ↓
Error, slice, and sensitivity analysis
      ↓
Artifacts, limitations, recommendation
```

At every stage, distinguish calculation correctness, statistical validity, operational validity, and decision validity.

## Risk register

        | Risk | Mitigation |
        |---|---|
        | Data leakage through duplicates or augmentation | Split before augmentation and audit identity. |
| Training mode during evaluation | Call `eval()` and disable gradients. |
| Tiny synthetic task overgeneralized | Limit claims to the bundled pattern distribution. |

In [ ]:
from pathlib import Path
import sys
import json
import warnings
warnings.filterwarnings("ignore")

_candidates = [Path.cwd(), *Path.cwd().parents]
COURSE_ROOT = next((p for p in _candidates if (p / "datasets").exists()), Path.cwd())
DATA_DIR = COURSE_ROOT / "datasets"
ARTIFACT_DIR = COURSE_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)
sys.path.insert(0, str(COURSE_ROOT))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print(f"Course root: {COURSE_ROOT}")

## 1. Load, inspect, and split

The dataset has three classes: vertical, horizontal, and diagonal patterns.

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader,TensorDataset
from sklearn.metrics import confusion_matrix,classification_report
torch.manual_seed(42)

X=np.load(DATA_DIR/"images/simple_shapes_X.npy")
y=np.load(DATA_DIR/"images/simple_shapes_y.npy")
classes=np.array(["vertical","horizontal","diagonal"])
print(X.shape,y.shape,np.bincount(y))
fig,ax=plt.subplots(figsize=(3,3))
ax.imshow(X[2,0],cmap="gray")
ax.set(title=f"Example: {classes[y[2]]}")
ax.axis("off"); plt.show()

order=np.random.default_rng(42).permutation(len(X))
n_train=int(.7*len(X)); n_val=int(.15*len(X))
train_idx=order[:n_train]; val_idx=order[n_train:n_train+n_val]; test_idx=order[n_train+n_val:]

## 2. Model and loaders

Use two convolution blocks and a small classifier. Input/output shapes are asserted.

In [ ]:
train_loader=DataLoader(TensorDataset(torch.tensor(X[train_idx]),torch.tensor(y[train_idx])),batch_size=48,shuffle=True)
val_x=torch.tensor(X[val_idx]); val_y=torch.tensor(y[val_idx])
test_x=torch.tensor(X[test_idx]); test_y=torch.tensor(y[test_idx])
model=nn.Sequential(
    nn.Conv2d(1,8,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
    nn.Conv2d(8,16,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
    nn.Flatten(),nn.Linear(16*4*4,24),nn.ReLU(),nn.Dropout(.15),nn.Linear(24,3)
)
assert model(torch.tensor(X[:4])).shape==(4,3)
print(model)

## 3. Train with validation checkpoint

Retain the state with best validation loss rather than the last epoch.

In [ ]:
optimizer=torch.optim.AdamW(model.parameters(),lr=.004,weight_decay=1e-4)
loss_fn=nn.CrossEntropyLoss()
best_loss=float("inf"); best_state=None; history=[]
for epoch in range(18):
    model.train(); train_total=0
    for xb,yb in train_loader:
        optimizer.zero_grad()
        loss=loss_fn(model(xb),yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),5)
        optimizer.step()
        train_total+=loss.item()*len(xb)
    model.eval()
    with torch.no_grad():
        val_logits=model(val_x)
        val_loss=loss_fn(val_logits,val_y).item()
        val_acc=(val_logits.argmax(1)==val_y).float().mean().item()
    history.append([epoch+1,train_total/len(train_idx),val_loss,val_acc])
    if val_loss<best_loss:
        best_loss=val_loss
        best_state={k:v.detach().clone() for k,v in model.state_dict().items()}
model.load_state_dict(best_state)
history=pd.DataFrame(history,columns=["epoch","train_loss","val_loss","val_accuracy"])
display(history.tail())

## 4. Test and error analysis

Evaluate exactly once after restoring the selected checkpoint.

In [ ]:
model.eval()
with torch.no_grad():
    logits=model(test_x)
    pred=logits.argmax(1).numpy()
print(classification_report(test_y.numpy(),pred,target_names=classes,digits=3))
cm=confusion_matrix(test_y.numpy(),pred)
display(pd.DataFrame(cm,index=classes,columns=classes))
fig,ax=plt.subplots(figsize=(6,4))
ax.plot(history.epoch,history.train_loss,label="Train loss")
ax.plot(history.epoch,history.val_loss,label="Validation loss")
ax.set(title="CNN learning curves",xlabel="Epoch",ylabel="Cross-entropy")
ax.legend(); plt.show()

## 5. Checkpoint and serving metadata

Save state, architecture metadata, classes, and normalization assumptions.

In [ ]:
checkpoint={
    "model_state":model.state_dict(),
    "classes":classes.tolist(),
    "input_shape":[1,16,16],
    "value_range":[0.0,1.0],
    "test_accuracy":float((pred==test_y.numpy()).mean()),
    "seed":42,
}
path=ARTIFACT_DIR/"capstone_shapes_cnn.pt"
torch.save(checkpoint,path)
print(path,path.stat().st_size)

## Model/project card

Complete this before presenting the result:

| Field | Statement |
|---|---|
| Intended use | |
| Excluded use | |
| Data population and coverage | |
| Target/metric definition | |
| Evaluation split | |
| Baseline | |
| Primary result | |
| Known limitations | |
| Important subgroup behaviour | |
| Human review / abstention | |
| Monitoring | |
| Owner and review cadence | |

## Final reflection

1. Which result changed your initial belief?
2. Which assumption creates the largest residual risk?
3. What simpler alternative was competitive?
4. What evidence is still required before an operational decision?
5. What would you monitor first after release?

Re-run the notebook from a clean kernel and verify generated artifacts before considering the capstone complete.